# 01. Data collection — recording human gameplay

Play Super Mario Bros (`SuperMarioBros-1-1-v0`) yourself. The notebook records the video and your actions.

## Controls

| Key | Action | Can change? |
|---|---|---|
| `d` | Right | Yes |
| `a` | Left | Yes |
| `o` | Jump (A) | Yes |
| `p` | Run / Fire (B) | Yes |
| Space | Start an episode (start recording) | No |
| Esc | Stop data collection | No |

You can change the game keys with `KEY_BINDINGS` in the play loop cell
(for example, `"right": "l"` makes the `l` key move right). After you change it, run that cell again.
The controls shown at the bottom of the screen are also made from `KEY_BINDINGS`, so they update automatically.
Space and Esc control the recording, so you cannot change them.

`SIMPLE_MOVEMENT` has only 7 actions (do nothing / right / right+A / right+B / right+A+B / A / left),
so there are no keys for down (crouch) or up.

Run the cells from top to bottom. When you run the last cell, the game window opens and you can play.

## How collection works

1. When the window opens, the game is **waiting**. The screen shows the controls. The game does not move during this time.
2. Press **Space** to start an episode. Recording starts at the same time.
3. When Mario dies or reaches the goal, the episode is saved automatically. The center of the screen shows
   a **summary of the episode** (distance, coins, score, time used, number of frames, and the Skill score from 0 to 100).
   Press Space to start the next episode.
4. Press **Esc** to stop data collection. If you were in the middle of an episode,
   the data up to that point is also saved, with `truncated=True`.

The Skill score rates one episode: progress (40%) + clear (10%) + speed (20%) + in-game score (30%).
Just clearing the stage gives 50 points. You get closer to 100 by collecting coins, defeating enemies, and clearing quickly.
Speed and score only count for episodes that clear the stage. See `02_review_data.ipynb` for the details of the formula
and the reasons for the weights. The constants and the formula are defined in `competition_config.py` (do not edit it).

In [1]:
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py._image_viewer import ImageViewer
from nes_py.wrappers import JoypadSpace

# Competition rules shared with 02/04. Do not edit that file.
from competition_config import TARGET_FPS

ENV_ID = "SuperMarioBros-1-1-v0"

# The play window would be tiny at the NES resolution (256x240), so open it
# at RENDER_SCALE times that size.
RENDER_SCALE = 2


class InstructionImageViewer(ImageViewer):
    """An ImageViewer that can show on-screen instructions."""

    BAR_HEIGHT = 30

    def __init__(self, caption, height, width, **kwargs):
        super().__init__(caption, height + self.BAR_HEIGHT, width, **kwargs)
        self._overlay_cache = {}

    def _overlay(self, text, centered):
        """Cache the drawables; rebuilding them every frame would redo text layout."""
        from pyglet import shapes

        key = (text, centered)
        if key not in self._overlay_cache:
            lines = text.splitlines()
            font_size = 13 if centered else 10
            line_height = font_size + 9
            box_height = line_height * len(lines) + 8
            if centered:
                game_height = self._window.height - self.BAR_HEIGHT
                box_y = self.BAR_HEIGHT + (game_height - box_height) // 2
            else:
                box_y = 0
                box_height = self.BAR_HEIGHT
            box = shapes.Rectangle(0, box_y, self._window.width, box_height, color=(0, 0, 0))
            box.opacity = 190 if centered else 255
            labels = [
                self.pyglet.text.Label(
                    line,
                    font_size=font_size,
                    bold=True,
                    x=self._window.width // 2,
                    y=box_y + (box_height - line_height * len(lines)) // 2
                    + (len(lines) - 1 - i) * line_height
                    + 4,
                    anchor_x="center",
                    anchor_y="bottom",
                    color=(255, 255, 255, 255),
                )
                for i, line in enumerate(lines)
            ]
            self._overlay_cache[key] = (box, labels)
        return self._overlay_cache[key]

    def show(self, frame, instructions=None, headline=None):
        if len(frame.shape) != 3:
            raise ValueError("frame should have shape with only 3 dimensions")
        if not self.is_open:
            self.open()
        self._window.clear()
        self._window.switch_to()
        self._window.dispatch_events()
        image = self.pyglet.image.ImageData(
            frame.shape[1], frame.shape[0], "RGB", frame.tobytes(), pitch=frame.shape[1] * -3
        )
        image.blit(
            0,
            self.BAR_HEIGHT,
            width=self._window.width,
            height=self._window.height - self.BAR_HEIGHT,
        )
        for text, centered in ((instructions, False), (headline, True)):
            if not text:
                continue
            box, labels = self._overlay(text, centered)
            box.draw()
            for label in labels:
                label.draw()
        self._window.flip()


def action_name(action_idx: int) -> str:
    """Convert a SIMPLE_MOVEMENT index into a readable string."""
    buttons = SIMPLE_MOVEMENT[action_idx]
    return "_".join(buttons) if buttons else "NOOP"


env = JoypadSpace(gym_super_mario_bros.make(ENV_ID, render_mode="human"), SIMPLE_MOVEMENT)

# Play runs at the NES native 60fps; only the frames we save are decimated to
# TARGET_FPS (defined in competition_config.py, shared with 04).
RECORD_EVERY_N = round(env.metadata["render_fps"] / TARGET_FPS)

print("SIMPLE_MOVEMENT:", SIMPLE_MOVEMENT)
print(f"render_fps={env.metadata['render_fps']} -> saved fps={TARGET_FPS} (RECORD_EVERY_N={RECORD_EVERY_N})")
print("reset() ->", env.reset())

SIMPLE_MOVEMENT: [['NOOP'], ['right'], ['right', 'A'], ['right', 'B'], ['right', 'A', 'B'], ['A'], ['left']]
render_fps=60 -> saved fps=30 (RECORD_EVERY_N=2)
reset() -> (array([[[104, 136, 252],
        [104, 136, 252],
        [104, 136, 252],
        ...,
        [104, 136, 252],
        [104, 136, 252],
        [104, 136, 252]],

       [[104, 136, 252],
        [104, 136, 252],
        [104, 136, 252],
        ...,
        [104, 136, 252],
        [104, 136, 252],
        [104, 136, 252]],

       [[104, 136, 252],
        [104, 136, 252],
        [104, 136, 252],
        ...,
        [104, 136, 252],
        [104, 136, 252],
        [104, 136, 252]],

       ...,

       [[240, 208, 176],
        [228,  92,  16],
        [228,  92,  16],
        ...,
        [228,  92,  16],
        [228,  92,  16],
        [  0,   0,   0]],

       [[240, 208, 176],
        [228,  92,  16],
        [228,  92,  16],
        ...,
        [228,  92,  16],
        [  0,   0,   0],
        [  0,   0, 

In [3]:
# Recording helpers: buffer frames in memory while playing, then write one
# mp4 + CSV per episode when it ends
import csv
import time
from pathlib import Path

import cv2

from competition_config import START_TIME, X_GOAL, skill_score, skill_terms

DATA_DIR = Path("../data")
EPISODES_DIR = DATA_DIR / "episodes"
MANIFEST_PATH = DATA_DIR / "manifest.csv"
EPISODES_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_FIELDS = [
    "episode_id", "num_frames", "score", "x_pos_max", "coins",
    "flag_get", "time_remaining", "truncated", "timestamp",
]
ACTION_FIELDS = ["frame_idx", "action", "action_name"]

frames = []
actions = []
_step_counter = 0
_episode_count = 0

print(f"existing episodes: {len(list(EPISODES_DIR.glob('ep_*')))}")


def _new_episode_id() -> str:
    """Build an episode id from the recording time.

    Sequential numbering restarts at 0 once episodes are deleted, so old ids get
    reused and the manifest rows can no longer be matched to their folders.
    """
    base = time.strftime("ep_%Y%m%d_%H%M%S")
    episode_id, suffix = base, 1
    while (EPISODES_DIR / episode_id).exists():  # in case two episodes end within the same second
        episode_id = f"{base}_{suffix}"
        suffix += 1
    return episode_id


def _append_manifest_row(row: dict) -> None:
    file_exists = MANIFEST_PATH.exists()
    with open(MANIFEST_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def episode_summary(row: dict) -> str:
    """Format a saved episode for the screen and for the notebook log."""
    outcome = "CLEARED!" if row["flag_get"] else "NOT CLEARED"
    terms = skill_terms(row)
    return "\n".join([
        f"Episode {row['episode_id']} saved",
        f"{outcome}    distance {row['x_pos_max']}/{X_GOAL} ({terms['progress']:.0%})    "
        f"coins {row['coins']}",
        f"score {row['score']}    time used {START_TIME - row['time_remaining']}    "
        f"frames {row['num_frames']}",
        f"SKILL {skill_score(row):.1f} / 100",
        f"(progress {terms['progress']:.0%}  clear {terms['clear']:.0%}  "
        f"speed {terms['speed']:.0%}  points {terms['points']:.0%})",
    ])


def _flush_episode(truncated: bool):
    """Write the buffered frames as a video + CSV and append a row to manifest.csv.

    Returns the episode summary so the play loop can show it on screen.
    """
    global _step_counter, _episode_count
    if not frames:
        return None

    ep_dir = EPISODES_DIR / _new_episode_id()
    ep_dir.mkdir(parents=True, exist_ok=True)

    height, width = frames[0].shape[0], frames[0].shape[1]
    video_writer = cv2.VideoWriter(
        str(ep_dir / "video.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), TARGET_FPS, (width, height)
    )
    for frame in frames:
        video_writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    video_writer.release()

    with open(ep_dir / "actions.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=ACTION_FIELDS)
        writer.writeheader()
        for i, action in enumerate(actions):
            writer.writerow({"frame_idx": i, "action": action, "action_name": action_name(action)})

    row = {
        "episode_id": ep_dir.name,
        "num_frames": len(frames),
        "score": env.unwrapped._score,
        "x_pos_max": env.unwrapped._x_position_max,
        "coins": env.unwrapped._coins,
        "flag_get": env.unwrapped._flag_get,
        "time_remaining": env.unwrapped._time,
        "truncated": truncated,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    _append_manifest_row(row)

    summary = episode_summary(row)
    if truncated:
        # an episode cut short with ESC is not a fair attempt, so its score is not comparable
        summary += "\n(stopped early with ESC - not counted as a real attempt)"
    print(summary, end="\n\n")

    _episode_count += 1
    frames.clear()
    actions.clear()
    _step_counter = 0
    return summary


def recording_callback(state, action, reward, done, next_state):
    """Buffer one step; on the last step of an episode, save it and return its summary."""
    global _step_counter
    if _step_counter % RECORD_EVERY_N == 0:
        # The frame returned by env.step()/reset() is a view into the emulator's
        # reusable buffer, so without a copy every entry would end up pointing at
        # the same (last) frame
        frames.append(state.copy())
        actions.append(action)
    _step_counter += 1
    if done:
        return _flush_episode(truncated=False)
    return None

existing episodes: 0


In [4]:
from pyglet import clock

NOOP_ACTION = 0
SPACE_KEY = ord(" ")

# --- Key bindings: edit the letters to remap the controls ------------------
# SPACE (start an episode) and ESC (quit) are fixed and cannot be reassigned.
# Only the buttons that SIMPLE_MOVEMENT actually uses need a key; that action
# list has no "down"/"up" action, so there is nothing to bind for ducking.
KEY_BINDINGS = {
    "right": "d",
    "left": "a",
    "A": "o",  # jump
    "B": "p",  # run / fire
}
# ---------------------------------------------------------------------------


def build_keys_to_action(actions, bindings):
    """Build a {pressed keys -> action index} map from the given key bindings.

    nes_py has its own hard-coded keyboard layout; we build the map ourselves so
    the bindings above are the single place that decides which key does what.
    """
    needed = {button for combo in actions for button in combo} - {"NOOP"}
    missing = sorted(needed - set(bindings))
    if missing:
        raise ValueError(f"KEY_BINDINGS has no key for: {missing}")
    for button, key in bindings.items():
        if len(key) != 1 or not key.isascii() or not key.isalpha():
            raise ValueError(
                f"KEY_BINDINGS[{button!r}] must be a single ASCII letter, got {key!r}"
            )
    keys = {button: key.lower() for button, key in bindings.items()}
    reused = sorted({k for k in keys.values() if list(keys.values()).count(k) > 1})
    if reused:
        raise ValueError(f"the same key is bound to more than one button: {reused}")
    return {
        tuple(sorted(ord(keys[button]) for button in combo if button != "NOOP")): action
        for action, combo in enumerate(actions)
    }


# Built from KEY_BINDINGS so the on-screen text can never drift from the bindings
CONTROLS_TEXT = (
    f"{KEY_BINDINGS['left']}/{KEY_BINDINGS['right']}: Move    "
    f"{KEY_BINDINGS['A']}: Jump (A)    "
    f"{KEY_BINDINGS['B']}: Run (B)    ESC: Quit"
)
RECORDING_TEXT = "REC  -  " + CONTROLS_TEXT
FIRST_HEADLINE = "Press SPACE to start episode"
NEXT_HEADLINE = "Press SPACE to start the next episode"


def waiting_headline(summary):
    """The centred message shown while waiting: last episode's summary, then the prompt."""
    if summary is None:
        return FIRST_HEADLINE
    return f"{summary}\n\n{NEXT_HEADLINE}"


def play_human_gated(env, callback=None):
    """Human play loop: SPACE starts an episode, ESC ends the session.

    The game is not stepped while waiting, so the player can get ready before the
    episode begins. When an episode ends (death or flag) its summary is shown on
    screen and the loop returns to waiting.
    """
    keys_to_action = build_keys_to_action(SIMPLE_MOVEMENT, KEY_BINDINGS)
    relevant_keys = {key for combo in keys_to_action for key in combo}
    # SPACE is not bound to any game input, so it is safe to use as the start trigger
    relevant_keys.add(SPACE_KEY)

    viewer = InstructionImageViewer(
        env.spec.id if env.spec is not None else env.__class__.__name__,
        env.observation_space.shape[0] * RENDER_SCALE,
        env.observation_space.shape[1] * RENDER_SCALE,
        monitor_keyboard=True,
        relevant_keys=relevant_keys,
    )

    target_frame_duration = 1 / env.metadata["render_fps"]
    last_frame_time = 0
    waiting = True
    last_summary = None  # summary of the episode that just finished, None before the first one
    state, _ = env.reset()

    try:
        while not viewer.is_escape_pressed:
            current_frame_time = time.time()
            if last_frame_time + target_frame_duration > current_frame_time:
                continue
            last_frame_time = current_frame_time
            clock.tick()

            pressed_keys = viewer.pressed_keys

            if waiting:
                viewer.show(
                    env.unwrapped.screen, CONTROLS_TEXT, headline=waiting_headline(last_summary)
                )
                if SPACE_KEY in pressed_keys:
                    waiting = False
                continue

            action = keys_to_action.get(
                tuple(key for key in pressed_keys if key != SPACE_KEY), NOOP_ACTION
            )
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = bool(terminated or truncated)
            viewer.show(env.unwrapped.screen, RECORDING_TEXT)

            if callback is not None:
                result = callback(state, action, reward, done, next_state)
                if done:
                    # the callback saves the episode and hands back its summary
                    last_summary = result

            state = next_state
            if done:
                state, _ = env.reset()
                waiting = True
    except KeyboardInterrupt:
        pass
    finally:
        viewer.close()
        env.close()


print("key bindings:", KEY_BINDINGS)
print("keys -> action:", build_keys_to_action(SIMPLE_MOVEMENT, KEY_BINDINGS))

key bindings: {'right': 'd', 'left': 'a', 'A': 'o', 'B': 'p'}
keys -> action: {(): 0, (100,): 1, (100, 111): 2, (100, 112): 3, (100, 111, 112): 4, (111,): 5, (97,): 6}


In [5]:
try:
    play_human_gated(env, callback=recording_callback)
finally:
    # If play ended early (ESC), save whatever is still buffered as truncated=True
    _flush_episode(truncated=True)

Episode ep_20261020_203519 saved
NOT CLEARED    distance 1416/3161 (45%)    coins 0
score 1350    time used 67    frames 706
SKILL 17.9 / 100
(progress 45%  clear 0%  speed 0%  points 0%)

Episode ep_20261020_203551 saved
NOT CLEARED    distance 2024/3161 (64%)    coins 1
score 2600    time used 87    frames 933
SKILL 25.6 / 100
(progress 64%  clear 0%  speed 0%  points 0%)

